# Phase 1-2 Re-Validation (v69 Fixes)

**v69 State Machine Fixes (Dec 30, 2024):**

1. **hasBrokenOut at detection** — Set when breakout detected, not when trade executes
2. **Unconditional consume** — Pending cleared at end of bar (taken OR skipped)
3. **backInside with guards** — Re-arm only when truly inside ORB, with directional guards

This notebook validates Python matches Pine v69.

In [1]:
# Cell 1: Setup and Data Fetch
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

import pandas as pd
import numpy as np
from dotenv import load_dotenv
load_dotenv(r'C:\Users\phemm\orb_lab\.env')

from data_collector import PolygonDataCollector

# Fetch with enough history
collector = PolygonDataCollector()
df = collector.fetch_bars('AMD', days_back=120, bar_size=1)
print(f"✓ Loaded {len(df)} bars")
print(f"Date range: {df.index[0].date()} to {df.index[-1].date()}")

# Available dates for testing
unique_dates = sorted(set(df.index.date))
print(f"\nAvailable trading days: {len(unique_dates)}")
print(f"Recent dates: {unique_dates[-5:]}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 120 days)...
  ✓ Loading from cache (7.6 hours old)
✓ Loaded 32304 bars
Date range: 2025-09-02 to 2025-12-30

Available trading days: 84
Recent dates: [datetime.date(2025, 12, 23), datetime.date(2025, 12, 24), datetime.date(2025, 12, 26), datetime.date(2025, 12, 29), datetime.date(2025, 12, 30)]


In [2]:
# Cell 2: Basic calculations (ATR)

def calc_atr(df, period=14):
    """ATR using Wilder's smoothing (RMA)"""
    df = df.copy()
    tr1 = df['high'] - df['low']
    tr2 = abs(df['high'] - df['close'].shift(1))
    tr3 = abs(df['low'] - df['close'].shift(1))
    df['tr'] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    df['atr'] = df['tr'].ewm(alpha=1/period, adjust=False).mean()
    return df

df = calc_atr(df)
print("✓ ATR calculated")

✓ ATR calculated


In [3]:
# Cell 3: v69 State Machine Implementation

def detect_orb_with_v69_state_machine(df, orb_minutes=5, 
                                       breakout_threshold_mult=0.1,
                                       min_body_strength=0.5,
                                       require_open_in_range=True):
    """
    Pine v69 State Machine:
    1. DETECT: Only on actual breakout candle (all criteria must pass)
    2. LATCH: Set pending=True AND hasBrokenOut=True at detection
    3. USE: baseBreakout = pending
    4. CONSUME: Clear pending at end of bar (taken OR skipped)
    5. RE-ARM: Reset hasBrokenOut only when backInside (both bounds)
    """
    df = df.copy()
    
    # ORB Session detection
    def is_in_orb(ts):
        bar_total = ts.hour * 60 + ts.minute
        return 9*60+30 <= bar_total < 9*60+30+orb_minutes
    
    df['in_session'] = df.index.map(is_in_orb)
    df['is_new_session'] = df['in_session'] & ~df['in_session'].shift(1, fill_value=False)
    
    # Calculate ORB levels
    df['orb_high'] = np.nan
    df['orb_low'] = np.nan
    current_high, current_low = np.nan, np.nan
    
    for i in range(len(df)):
        if df['is_new_session'].iloc[i]:
            current_high = df['high'].iloc[i]
            current_low = df['low'].iloc[i]
        elif df['in_session'].iloc[i]:
            current_high = max(current_high, df['high'].iloc[i])
            current_low = min(current_low, df['low'].iloc[i])
        df.iloc[i, df.columns.get_loc('orb_high')] = current_high
        df.iloc[i, df.columns.get_loc('orb_low')] = current_low
    
    df['orb_complete'] = ~df['in_session'] & df['orb_high'].notna() & df['orb_low'].notna()
    
    # Breakout quality checks
    df['breakout_threshold'] = df['atr'] * breakout_threshold_mult
    df['sufficient_high'] = df['close'] > (df['orb_high'] + df['breakout_threshold'])
    df['sufficient_low'] = df['close'] < (df['orb_low'] - df['breakout_threshold'])
    
    df['candle_body'] = abs(df['close'] - df['open'])
    df['candle_range'] = df['high'] - df['low']
    df['body_ratio'] = df['candle_body'] / df['candle_range'].replace(0, np.nan)
    df['strong_body'] = df['body_ratio'] >= min_body_strength
    
    # STATE MACHINE COLUMNS
    df['new_long_breakout'] = False
    df['new_short_breakout'] = False
    df['long_breakout_pending'] = False
    df['short_breakout_pending'] = False
    df['has_broken_out_high'] = False
    df['has_broken_out_low'] = False
    df['back_inside'] = False
    df['state_event'] = ''
    
    # State variables
    has_broken_high = False
    has_broken_low = False
    
    for i in range(len(df)):
        row = df.iloc[i]
        events = []
        
        # Reset at new session
        if row['is_new_session']:
            has_broken_high = False
            has_broken_low = False
            events.append('NEW_DAY_RESET')
        
        orb_complete = row['orb_complete']
        orb_high = row['orb_high']
        orb_low = row['orb_low']
        close_price = row['close']
        
        # STEP 1: Check backInside for re-arm
        back_inside = close_price <= orb_high and close_price >= orb_low
        df.iloc[i, df.columns.get_loc('back_inside')] = back_inside
        
        if orb_complete and back_inside:
            if has_broken_high:
                has_broken_high = False
                events.append('RE-ARM_HIGH')
            if has_broken_low:
                has_broken_low = False
                events.append('RE-ARM_LOW')
        
        # STEP 2: DETECT new breakouts
        new_long = False
        new_short = False
        long_pending_this_bar = False
        short_pending_this_bar = False
        
        if orb_complete:
            open_price = row['open']
            sufficient_high = row['sufficient_high']
            sufficient_low = row['sufficient_low']
            strong_body = row['strong_body']
            
            if require_open_in_range:
                open_in_range = (open_price >= orb_low) and (open_price <= orb_high)
                new_long = open_in_range and (close_price > orb_high) and sufficient_high and strong_body
                new_short = open_in_range and (close_price < orb_low) and sufficient_low and strong_body
            else:
                new_long = (close_price > orb_high) and sufficient_high and strong_body
                new_short = (close_price < orb_low) and sufficient_low and strong_body
        
        # STEP 3: LATCH + set hasBrokenOut at detection (v69 fix)
        if new_long and not has_broken_high:
            long_pending_this_bar = True
            has_broken_high = True  # v69: set at detection, not execution
            events.append('DETECT_LONG')
            events.append('LATCH: pending=T, hasBroken=T')
        
        if new_short and not has_broken_low:
            short_pending_this_bar = True
            has_broken_low = True  # v69: set at detection, not execution
            events.append('DETECT_SHORT')
            events.append('LATCH: pending=T, hasBroken=T')
        
        # STEP 4: CONSUME pending at end of bar (v69 fix)
        # In real backtester, entry logic happens here, then consume
        # For this debug, we just show pending was true this bar
        if long_pending_this_bar:
            events.append('CONSUME: pending cleared')
        if short_pending_this_bar:
            events.append('CONSUME: pending cleared')
        
        # Store state
        df.iloc[i, df.columns.get_loc('new_long_breakout')] = new_long
        df.iloc[i, df.columns.get_loc('new_short_breakout')] = new_short
        df.iloc[i, df.columns.get_loc('long_breakout_pending')] = long_pending_this_bar
        df.iloc[i, df.columns.get_loc('short_breakout_pending')] = short_pending_this_bar
        df.iloc[i, df.columns.get_loc('has_broken_out_high')] = has_broken_high
        df.iloc[i, df.columns.get_loc('has_broken_out_low')] = has_broken_low
        df.iloc[i, df.columns.get_loc('state_event')] = ' | '.join(events) if events else ''
    
    return df

print("✓ v69 State Machine function defined")

✓ v69 State Machine function defined


In [4]:
# Cell 4: Process data with v69 state machine
df = detect_orb_with_v69_state_machine(df)
print("✓ State machine applied")

# Count breakouts detected
long_breakouts = df['new_long_breakout'].sum()
short_breakouts = df['new_short_breakout'].sum()
print(f"\nBreakouts detected: {long_breakouts} LONG, {short_breakouts} SHORT")

✓ State machine applied

Breakouts detected: 199 LONG, 203 SHORT


In [5]:
# Cell 5: Pick test dates dynamically

# Find days with breakouts
days_with_long = df[df['new_long_breakout']].index.date
days_with_short = df[df['new_short_breakout']].index.date

print("Days with LONG breakouts (last 5):")
print(sorted(set(days_with_long))[-5:])

print("\nDays with SHORT breakouts (last 5):")
print(sorted(set(days_with_short))[-5:])

# Pick recent test dates
if len(days_with_long) > 0:
    test_date_long = sorted(set(days_with_long))[-1]
    print(f"\n→ Using {test_date_long} for LONG test")
if len(days_with_short) > 0:
    test_date_short = sorted(set(days_with_short))[-1]
    print(f"→ Using {test_date_short} for SHORT test")

Days with LONG breakouts (last 5):
[datetime.date(2025, 12, 18), datetime.date(2025, 12, 19), datetime.date(2025, 12, 26), datetime.date(2025, 12, 29), datetime.date(2025, 12, 30)]

Days with SHORT breakouts (last 5):
[datetime.date(2025, 12, 17), datetime.date(2025, 12, 18), datetime.date(2025, 12, 22), datetime.date(2025, 12, 26), datetime.date(2025, 12, 30)]

→ Using 2025-12-30 for LONG test
→ Using 2025-12-30 for SHORT test


In [6]:
# Cell 6: TEST 1 - State Machine Flow (LONG)

print("="*70)
print(f"TEST 1: State Machine Flow - {test_date_long}")
print("="*70)

day_df = df[df.index.date == test_date_long].copy()

print(f"\nORB High: {day_df['orb_high'].iloc[-1]:.2f}")
print(f"ORB Low: {day_df['orb_low'].iloc[-1]:.2f}")

# Find the breakout
breakouts = day_df[day_df['new_long_breakout']]
if len(breakouts) > 0:
    first_breakout = breakouts.iloc[0]
    ts = breakouts.index[0]
    
    print(f"\n📍 LONG BREAKOUT detected at {ts.strftime('%H:%M')}")
    print(f"   Open:  {first_breakout['open']:.2f}")
    print(f"   Close: {first_breakout['close']:.2f}")
    print(f"   ORB High: {first_breakout['orb_high']:.2f}")
    
    open_in_range = first_breakout['open'] >= first_breakout['orb_low'] and first_breakout['open'] <= first_breakout['orb_high']
    print(f"   Open in range: {open_in_range}")
    print(f"   Sufficient threshold: {first_breakout['sufficient_high']}")
    print(f"   Strong body: {first_breakout['strong_body']} (ratio: {first_breakout['body_ratio']:.2f})")
    print(f"\n   State Event: {first_breakout['state_event']}")
    print(f"   hasBrokenOutHigh: {first_breakout['has_broken_out_high']}")
    
    # Check next few bars - should NOT fire again
    print("\n--- Next 3 bars (should NOT have new breakouts) ---")
    next_bars = day_df.loc[ts:].iloc[1:4]
    for idx, row in next_bars.iterrows():
        print(f"   {idx.strftime('%H:%M')}: new_long={row['new_long_breakout']}, hasBroken={row['has_broken_out_high']}, event='{row['state_event']}'")
else:
    print("❌ No LONG breakout found on this date")

TEST 1: State Machine Flow - 2025-12-30

ORB High: 216.64
ORB Low: 215.00

📍 LONG BREAKOUT detected at 11:22
   Open:  216.44
   Close: 216.69
   ORB High: 216.64
   Open in range: True
   Sufficient threshold: True
   Strong body: True (ratio: 0.70)

   State Event: DETECT_LONG | LATCH: pending=T, hasBroken=T | CONSUME: pending cleared
   hasBrokenOutHigh: True

--- Next 3 bars (should NOT have new breakouts) ---
   11:23: new_long=False, hasBroken=False, event='RE-ARM_HIGH'
   11:24: new_long=False, hasBroken=False, event=''
   11:25: new_long=False, hasBroken=False, event=''


In [7]:
# Cell 7: TEST 2 - State Machine Flow (SHORT)

print("="*70)
print(f"TEST 2: State Machine Flow - {test_date_short}")
print("="*70)

day_df = df[df.index.date == test_date_short].copy()

print(f"\nORB High: {day_df['orb_high'].iloc[-1]:.2f}")
print(f"ORB Low: {day_df['orb_low'].iloc[-1]:.2f}")

# Find the breakout
breakouts = day_df[day_df['new_short_breakout']]
if len(breakouts) > 0:
    first_breakout = breakouts.iloc[0]
    ts = breakouts.index[0]
    
    print(f"\n📍 SHORT BREAKOUT detected at {ts.strftime('%H:%M')}")
    print(f"   Open:  {first_breakout['open']:.2f}")
    print(f"   Close: {first_breakout['close']:.2f}")
    print(f"   ORB Low: {first_breakout['orb_low']:.2f}")
    
    open_in_range = first_breakout['open'] >= first_breakout['orb_low'] and first_breakout['open'] <= first_breakout['orb_high']
    print(f"   Open in range: {open_in_range}")
    print(f"   Sufficient threshold: {first_breakout['sufficient_low']}")
    print(f"   Strong body: {first_breakout['strong_body']} (ratio: {first_breakout['body_ratio']:.2f})")
    print(f"\n   State Event: {first_breakout['state_event']}")
    print(f"   hasBrokenOutLow: {first_breakout['has_broken_out_low']}")
    
    # Check next few bars - should NOT fire again
    print("\n--- Next 3 bars (should NOT have new breakouts) ---")
    next_bars = day_df.loc[ts:].iloc[1:4]
    for idx, row in next_bars.iterrows():
        print(f"   {idx.strftime('%H:%M')}: new_short={row['new_short_breakout']}, hasBroken={row['has_broken_out_low']}, event='{row['state_event']}'")
else:
    print("❌ No SHORT breakout found on this date")

TEST 2: State Machine Flow - 2025-12-30

ORB High: 216.64
ORB Low: 215.00

📍 SHORT BREAKOUT detected at 10:25
   Open:  215.06
   Close: 214.94
   ORB Low: 215.00
   Open in range: True
   Sufficient threshold: True
   Strong body: True (ratio: 0.60)

   State Event: DETECT_SHORT | LATCH: pending=T, hasBroken=T | CONSUME: pending cleared
   hasBrokenOutLow: True

--- Next 3 bars (should NOT have new breakouts) ---
   10:26: new_short=False, hasBroken=False, event='RE-ARM_LOW'
   10:27: new_short=False, hasBroken=False, event=''
   10:28: new_short=False, hasBroken=False, event=''


In [8]:
# Cell 8: TEST 3 - Phantom Prevention
# Verify: bars after breakout that don't open in range should NOT trigger

print("="*70)
print("TEST 3: Phantom Entry Prevention")
print("="*70)

# Find all bars closing outside ORB after a breakout
# They should NOT be marked as new breakouts if open not in range

day_df = df[df.index.date == test_date_long].copy()
post_breakout = day_df[day_df['orb_complete']]

# Find bars that close above ORB but are NOT marked as breakouts
above_orb = post_breakout[post_breakout['close'] > post_breakout['orb_high']]
not_breakouts = above_orb[~above_orb['new_long_breakout']]

print(f"\nBars closing above ORB: {len(above_orb)}")
print(f"Bars NOT marked as breakouts: {len(not_breakouts)}")

if len(not_breakouts) > 0:
    print("\n--- Non-breakout bars above ORB (correctly blocked) ---")
    for idx, row in not_breakouts.head(5).iterrows():
        open_in_range = row['open'] >= row['orb_low'] and row['open'] <= row['orb_high']
        print(f"   {idx.strftime('%H:%M')}: open_in_range={open_in_range}, hasBroken={row['has_broken_out_high']}")
    print("\n✓ Phantom entries correctly blocked")
else:
    print("\n(No phantom candidates found - may need different test date)")

TEST 3: Phantom Entry Prevention

Bars closing above ORB: 2
Bars NOT marked as breakouts: 0

(No phantom candidates found - may need different test date)


In [9]:
# Cell 9: TEST 4 - Re-arm Test
# Find a day where price returns inside ORB after breaking out

print("="*70)
print("TEST 4: Re-arm When Price Returns Inside ORB")
print("="*70)

# Look for RE-ARM events
rearm_events = df[df['state_event'].str.contains('RE-ARM', na=False)]

if len(rearm_events) > 0:
    print(f"\nFound {len(rearm_events)} RE-ARM events")
    print("\nSample re-arm events:")
    for idx, row in rearm_events.head(3).iterrows():
        print(f"   {idx}: {row['state_event']}")
        print(f"      close={row['close']:.2f}, orb_high={row['orb_high']:.2f}, orb_low={row['orb_low']:.2f}")
        print(f"      back_inside={row['back_inside']}")
else:
    print("\nNo RE-ARM events found in data (price never returned inside ORB)")
    print("This is normal if trends are strong.")

TEST 4: Re-arm When Price Returns Inside ORB

Found 344 RE-ARM events

Sample re-arm events:
   2025-09-03 11:22:00-04:00: RE-ARM_HIGH
      close=161.69, orb_high=161.91, orb_low=160.58
      back_inside=True
   2025-09-03 12:15:00-04:00: RE-ARM_HIGH
      close=161.85, orb_high=161.91, orb_low=160.58
      back_inside=True
   2025-09-03 12:23:00-04:00: RE-ARM_HIGH
      close=161.91, orb_high=161.91, orb_low=160.58
      back_inside=True


In [10]:
# Cell 10: Run full backtester

from orb_backtester import ORBBacktester, ORBConfig

config = ORBConfig(
    orb_minutes=5,
    require_open_in_range=True,
    breakout_threshold_atr=0.1,
    min_body_strength_ratio=0.5,
    min_acceptable_rr=1.5,
    enable_confluence_scoring=True,
    min_confluence_score=3,
    limit_trading_to_first_hour=True,
    enable_long_entries=True,
    enable_short_entries=True,
)

# Re-fetch clean data for backtester
df_clean = collector.fetch_bars('AMD', days_back=120, bar_size=1)

# Debug: Check values for Dec 29 09:36 bar
from datetime import date
backtester = ORBBacktester(config)
dec29 = df_clean[df_clean.index.date == date(2025, 12, 29)]
bar_0936 = dec29.between_time('09:36', '09:36')
if len(bar_0936) > 0:
    idx = df_clean.index.get_loc(bar_0936.index[0])
    indicators = backtester._calculate_indicators(df_clean)
    
    print("=== DEBUG: Dec 29 09:36 bar ===")
    print(f"Close: {bar_0936['close'].iloc[0]:.2f}")
    print(f"ATR: {indicators['atr'].iloc[idx]:.4f}")
    print(f"Vol Factor: {indicators['vol_factor'].iloc[idx]:.2f}")
    print(f"Vol State: {indicators['vol_state'].iloc[idx]}")
    print(f"VWAP: {indicators['vwap'].iloc[idx]:.2f}")

# Run backtest
trades = backtester.run(df_clean, symbol='AMD')

print(f"\nTotal trades: {len(trades)}")

if len(trades) > 0:
    print("\n--- Recent Trades ---")
    for idx, trade in trades.tail(10).iterrows():
        print(f"\n{trade['entry_time']} | {trade['direction']}")
        print(f"   Entry: ${trade['entry_price']:.2f} | Exit: ${trade['exit_price']:.2f}")
        print(f"   P&L: ${trade['pnl']:.2f} ({trade['pnl_r']:.2f}R) | {trade['exit_reason']}")

📊 Fetching AMD (1-min bars, 120 days)...
  ✓ Loading from cache (7.6 hours old)
=== DEBUG: Dec 29 09:36 bar ===
Close: 213.20
ATR: 0.9660
Vol Factor: 1.22
Vol State: NORMAL
VWAP: 212.14

Total trades: 82

--- Recent Trades ---

2025-12-16 10:07:00-05:00 | LONG
   Entry: $208.84 | Exit: $208.83
   P&L: $-200.00 (-1.00R) | STOP_LOSS

2025-12-17 09:35:00-05:00 | SHORT
   Entry: $208.56 | Exit: $207.32
   P&L: $177.76 (0.89R) | TRAILING_STOP

2025-12-18 10:16:00-05:00 | SHORT
   Entry: $203.45 | Exit: $204.55
   P&L: $-200.00 (-1.00R) | STOP_LOSS

2025-12-18 10:28:00-05:00 | LONG
   Entry: $206.31 | Exit: $206.30
   P&L: $-200.00 (-1.00R) | STOP_LOSS

2025-12-19 09:39:00-05:00 | LONG
   Entry: $207.57 | Exit: $207.56
   P&L: $-200.00 (-1.00R) | STOP_LOSS

2025-12-22 09:58:00-05:00 | SHORT
   Entry: $214.81 | Exit: $214.82
   P&L: $-200.00 (-1.00R) | STOP_LOSS

2025-12-26 09:43:00-05:00 | SHORT
   Entry: $213.85 | Exit: $213.86
   P&L: $-200.00 (-1.00R) | STOP_LOSS

2025-12-26 09:50:00-05:0

In [ ]:
# Cell 11: Validation Checklist

print("""
═══════════════════════════════════════════════════════════════
                    VALIDATION CHECKLIST v69
═══════════════════════════════════════════════════════════════

v69 State Machine Fixes:
────────────────────────
□ hasBrokenOut set at DETECTION (not execution)
□ Pending consumed at end of bar (taken OR skipped)
□ Re-arm only when backInside (both bounds)
□ Directional guards prevent cross-contamination

Test Results:
─────────────
□ TEST 1: LONG breakout detected, pending consumed same bar
□ TEST 2: SHORT breakout detected, pending consumed same bar
□ TEST 3: Phantom entries blocked (bars not opening in range)
□ TEST 4: Re-arm works when price returns inside ORB

Compare with TradingView v69:
─────────────────────────────
□ Pick a trade from Cell 10
□ Replay in TradingView on same date/symbol
□ Verify entry time matches
□ Verify entry price matches
□ Verify no phantom late entries

═══════════════════════════════════════════════════════════════
""")

In [11]:
# Find break-even stops (exit at entry price, STOP_LOSS reason)
be_candidates = trades[
    (trades['exit_reason'] == 'STOP_LOSS') & 
    (abs(trades['exit_price'] - trades['entry_price']) < 0.05)
]
print(f"Potential BE stops: {len(be_candidates)}")
print(be_candidates[['entry_time', 'direction', 'entry_price', 'exit_price', 'pnl_r']])

Potential BE stops: 53
                  entry_time direction  entry_price  exit_price  pnl_r
3  2025-09-09 09:36:00-04:00      LONG     153.5700    153.5600   -1.0
4  2025-09-11 09:35:00-04:00     SHORT     157.9350    157.9450   -1.0
6  2025-09-12 09:48:00-04:00     SHORT     155.9000    155.9000    0.0
7  2025-09-15 10:09:00-04:00     SHORT     158.0900    158.1000   -1.0
8  2025-09-17 09:46:00-04:00      LONG     160.5500    160.5500    0.0
10 2025-09-24 09:46:00-04:00     SHORT     162.7500    162.7600   -1.0
14 2025-10-01 09:55:00-04:00      LONG     162.6000    162.6000    0.0
15 2025-10-02 09:41:00-04:00     SHORT     166.6700    166.6800   -1.0
16 2025-10-02 09:49:00-04:00     SHORT     166.7000    166.7100   -1.0
17 2025-10-03 09:40:00-04:00     SHORT     169.4900    169.5000   -1.0
18 2025-10-07 09:50:00-04:00     SHORT     213.9800    213.9900   -1.0
19 2025-10-07 09:55:00-04:00     SHORT     214.1800    214.1900   -1.0
20 2025-10-07 09:58:00-04:00     SHORT     214.2000   